## 掛載雲端硬碟

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 更改檔案所在路徑


In [2]:
# Change to your own folder !!!
%cd /content/drive/MyDrive/EAI_Lab2/

/content/drive/.shortcut-targets-by-id/1Lh36c1zmP3wgq2NRLDf0V7JxMipjPBsi/EAI_Lab2


## 載入函式庫


In [3]:
import os

import torch
import torch.nn as nn
from torch.autograd import Variable
from torchvision import datasets, transforms
import numpy as np

from models.resnet import ResNet50

## 超參數設定

In [19]:
DATASET = 'cifar10'
TEST_BATCH_SIZE = 1000
CUDA = True
PRUNE_PERCENT = 0.9 # Change your prune ratio!
WEIGHT_PATH = '/content/drive/MyDrive/EAI_Lab2/model_best.pth' # Change to your own folder !!!
PRUNE_PATH = '/content/drive/MyDrive/EAI_Lab2/model_prune.pth' # Change to your own folder !!!

## 載入模型

In [20]:
CUDA = CUDA and torch.cuda.is_available()

model = ResNet50(num_classes=10)
if CUDA:
    model.cuda()

if WEIGHT_PATH:
    if os.path.isfile(WEIGHT_PATH):
        checkpoint = torch.load(WEIGHT_PATH)
        best_prec1 = checkpoint['best_prec1']
        model.load_state_dict(checkpoint['state_dict'])
        print('LOADING CHECKPOINT {} @EPOCH={}, BEST_PREC1={}'.format(WEIGHT_PATH,checkpoint['epoch'],best_prec1))

    else:
        print("NO CHECKPOINT FOUND")

print(model)

LOADING CHECKPOINT /content/drive/MyDrive/EAI_Lab2/model_best.pth @EPOCH=40, BEST_PREC1=0.9074000120162964
ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU

## 進行剪枝
#### 計算所有Batch Normalizaiton中的scale factor絕對值大小並排序
#### 利用設定好的PRUNE_PERCENT來取得閥值

In [21]:
total = 0
for m in model.modules():
    if isinstance(m, nn.BatchNorm2d):
        total += m.weight.data.shape[0]

bn = torch.zeros(total)
index = 0
for m in model.modules():
    if isinstance(m, nn.BatchNorm2d):
        size = m.weight.data.shape[0]
        bn[index:(index+size)] = m.weight.data.abs().clone()
        index += size

y, i = torch.sort(bn)

threshold_index = int(total * PRUNE_PERCENT)
threshold = y[threshold_index]


## 根據Batch Normalization Layer資訊建立CONFIG
#### 1. 複製Batch Normalization Layer的weight(也就是scale factor γ)
#### 2. 建立mask，大於threshold的index的值會設成1,小於threshold的值會設成0
#### 3. mask的值加總後，會是剪枝後Layer對應的輸出channel數
#### 4. 最後得到要建立剪枝模型的CONFIG

In [22]:
cfg = []
cfg_mask = []
pruned = 0
total = 0

In [23]:
min_keep = 3           # 每層最少保留通道數，避免被剪成 0
bn_idx = -1  # 用於判斷 bn1/bn2/bn3；假設：stem 之後每 3 個 BN 為一組

# ===== 1) 蒐集所有 BN 的 |γ|，計算 threshold =====
bn_weights = []
for m in model.modules():
    if isinstance(m, nn.BatchNorm2d):
        bn_weights.append(m.weight.data.abs().clone())

all_bn = torch.cat(bn_weights)
# 使用 quantile 取得門檻（比 threshold 大者保留）
threshold = torch.quantile(all_bn, PRUNE_PERCENT)


for k, m in enumerate((model.modules())):
    if isinstance(m, nn.BatchNorm2d):
        total += m.weight.data.shape[0]
        bn_idx += 1

        weight_copy = m.weight.data.clone()
        mask = weight_copy.abs().gt(threshold).float().cuda() # 大於 threshold 的設為 True (1.0)，其餘為 False(0.0)

        device = weight_copy.device

        # 注意: 需自行設計處理剩下channel數為0的情況 (e.g. 至少保留3個channel)
        ################################################
        #          請填空          #
        ################################################

        is_bn3 = (bn_idx >= 1) and ((bn_idx - 1) % 3 == 2)

        if is_bn3:
            # ===== 不剪 bn3：mask 全 1 =====
            mask = torch.ones_like(weight_copy, device=device)
        else:
            # 先依 threshold 建初始 mask
            mask = weight_copy.gt(threshold.to(device)).float()

            # 若剩餘通道太少（或 0），至少保留 |γ| 最大的 min_keep 個
            remain = int(mask.sum().item())
            if remain < min_keep:
                keep_k = min(min_keep, weight_copy.numel())
                topk_idx = torch.topk(weight_copy, k=keep_k, largest=True, sorted=False).indices
                mask.zero_()
                mask[topk_idx] = 1.0





        # 處理剪枝後的權重
        m.weight.data.mul_(mask)

        if m.bias is not None:
            m.bias.data.mul_(mask)
        kept = int(mask.sum().item())
        pruned += (mask.shape[0] - kept)
        cfg.append(kept)              # 這層 BN 剩下的通道數
        cfg_mask.append(mask.clone()) # 這層的 mask（供之後複製 Conv/BN/FC 權重時索引用）

        #m.bias.data.mul_(mask)
        #pruned = pruned + mask.shape[0] - torch.sum(mask)
        #cfg.append(int(torch.sum(mask)))    # 記錄每一層 BN 剩下幾個通道
        #cfg_mask.append(mask.clone())     # 儲存每層對應的 mask
        print('layer index: {:d} \t total channel: {:d} \t remaining channel: {:d}'.
            format(k, mask.shape[0], int(torch.sum(mask))))

pruned_ratio = pruned/total

print(f'PRUNE RATIO={pruned_ratio}')
print('PREPROCESSING SUCCESSFUL!')

print(f'cfg: {cfg}')


layer index: 2 	 total channel: 64 	 remaining channel: 64
layer index: 8 	 total channel: 64 	 remaining channel: 64
layer index: 10 	 total channel: 64 	 remaining channel: 64
layer index: 12 	 total channel: 256 	 remaining channel: 256
layer index: 17 	 total channel: 64 	 remaining channel: 64
layer index: 19 	 total channel: 64 	 remaining channel: 64
layer index: 21 	 total channel: 256 	 remaining channel: 256
layer index: 25 	 total channel: 64 	 remaining channel: 64
layer index: 27 	 total channel: 64 	 remaining channel: 64
layer index: 29 	 total channel: 256 	 remaining channel: 256
layer index: 34 	 total channel: 128 	 remaining channel: 106
layer index: 36 	 total channel: 128 	 remaining channel: 117
layer index: 38 	 total channel: 512 	 remaining channel: 512
layer index: 43 	 total channel: 128 	 remaining channel: 120
layer index: 45 	 total channel: 128 	 remaining channel: 124
layer index: 47 	 total channel: 512 	 remaining channel: 512
layer index: 51 	 total 

## 建立剪枝模型

In [24]:
newmodel = ResNet50(num_classes=10, cfg=cfg)
newmodel.cuda()

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
    )
   

### 將原本的模型權重複製到剪枝的模型
#### 根據不同層決定要複製什麼權重
###### Batch Normalization Layer
1.   scale factor
2.   bias
3.   running mean
4.   running variance

###### Convolutional Layer
1.   weight

###### Linear Layer
1.   weight
2.   bias




In [25]:
old_modules = list(model.modules())
new_modules = list(newmodel.modules())

layer_id_in_cfg = 0
start_mask = torch.ones(3) #3為input channel(R,G,B)
end_mask = cfg_mask[layer_id_in_cfg]
bn_count = 0

for layer_id in range(len(old_modules)):

    m0 = old_modules[layer_id]
    m1 = new_modules[layer_id]

    if isinstance(m0, nn.BatchNorm2d):
        bn_count += 1

        #### 找出遮罩中非零元素的index ####
        ################################################
        #          請填空          #
        ################################################
        idx = torch.nonzero(cfg_mask[layer_id_in_cfg]).squeeze().cpu().numpy()
        if idx.ndim == 0:
            idx = idx[None]


        #### 複製weight, bias, running mean,and running variance ####
        ################################################
        #          請填空          #
        ################################################
        m1.weight.data = m0.weight.data[idx].clone()
        if m1.bias is not None:
            m1.bias.data   = m0.bias.data[idx].clone()
        m1.running_mean  = m0.running_mean[idx].clone()
        m1.running_var   = m0.running_var[idx].clone()



        layer_id_in_cfg += 1
        start_mask = end_mask.clone()

        #最後一層連接層不做修改
        if layer_id_in_cfg < len(cfg_mask):
            end_mask = cfg_mask[layer_id_in_cfg]


    elif isinstance(m0, nn.Conv2d):
        if isinstance(old_modules[layer_id + 1], nn.BatchNorm2d):
            idx0 = np.squeeze(np.argwhere(np.asarray(start_mask.cpu().numpy())))
            idx1 = np.squeeze(np.argwhere(np.asarray(end_mask.cpu().numpy())))

            #### 複製weight ####
            ################################################
            #          請填空          #
            ################################################
            if idx0.ndim == 0: idx0 = idx0[None]
            if idx1.ndim == 0: idx1 = idx1[None]

            w = m0.weight.data[:, idx0, :, :].clone()  # select input
            w = w[idx1, :, :, :].clone()        # select output
            m1.weight.data = w.clone()


        # downsample 層不用prune
        else:
            m1.weight.data = m0.weight.data.clone()


    elif isinstance(m0, nn.Linear):

        idx0 = np.squeeze(np.argwhere(np.asarray(start_mask.cpu().numpy())))
        if idx0.ndim == 0:
            idx0 = np.expand_dims(idx0, 0)

        #### 複製weight ####
        ################################################
        #          請填空          #
        ################################################
        m1.weight.data = m0.weight.data[:, idx0].clone()

        #### 複製bias ####
        m1.bias.data = m0.bias.data.clone()


## 測試函數




In [26]:
def test(model):
    kwargs = {'num_workers': 1, 'pin_memory': True} if CUDA else {}
    test_loader = torch.utils.data.DataLoader(
        datasets.CIFAR10('./data', train=False, download=True, transform=transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))])),
        batch_size=TEST_BATCH_SIZE, shuffle=True, **kwargs)

    model.eval()
    correct = 0
    with torch.no_grad():
      for data, target in test_loader:
          if CUDA:
              data, target = data.cuda(), target.cuda()
          data, target = Variable(data), Variable(target)
          output = model(data)
          pred = output.data.max(1, keepdim=True)[1]
          correct += pred.eq(target.data.view_as(pred)).cpu().sum()

    print('\nTest set: Accuracy: {}/{} ({:.1f}%)\n'.format(
        correct, len(test_loader.dataset), 100. * correct / len(test_loader.dataset)))
    return correct / float(len(test_loader.dataset))

## 儲存模型並印出結果，以及剪枝後的test acc


In [27]:
torch.save({'cfg': cfg, 'state_dict': newmodel.state_dict()}, PRUNE_PATH)

print(newmodel)
model = newmodel.cuda()
test(model)

#額外加的剪枝比例
print('Prune percent(%):{:.1f}%'.format(PRUNE_PERCENT*100))

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
    )
   